In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from stochasticbatopt.plotting import plot_convexity_cross_section

## CONFIG

In [ ]:
INSPECT_DAY = 4     # day index for the cross-section
N_POINTS    = 60    # grid resolution along the cross-section
N_STD       = 3.0   # how many σ to extend the t-axis

## Load model

In [ ]:
with open('../sbo_cache/model_and_result.pkl', 'rb') as fh:
    state = pickle.load(fh)

model      = state['model']
result     = state['result']
scenarios  = state['scenarios']
time_index = state['time_index']
print(f'Loaded: {model.n_days} days, N={scenarios.shape[0]} scenarios')

## F cross-section plot

In [ ]:
%matplotlib inline
plot_convexity_cross_section(
    model,
    day      = INSPECT_DAY,
    n_points = N_POINTS,
    n_std    = N_STD,
)

## Jensen gap decomposition

Compute the day-level Jensen gap directly from scenario revenues.

In [ ]:
p  = model.params
h0 = INSPECT_DAY * 24
h1 = h0 + 24

prices_day = model.price_scenarios[:, h0:h1]
ein_day    = result['e_in'][:, h0:h1]
eout_day   = result['e_out'][:, h0:h1]

rev_scen = np.sum(
    prices_day * eout_day * p.discharge_efficiency
    - prices_day * ein_day  / p.charge_efficiency,
    axis=1,
) * p.time_step

ref     = model._reference
ref_rev = float(np.sum(
    ref['price_path'][h0:h1] * ref['e_out'][h0:h1] * p.discharge_efficiency
    - ref['price_path'][h0:h1] * ref['e_in'][h0:h1]  / p.charge_efficiency
) * p.time_step)

mean_rev   = float(rev_scen.mean())
jensen_gap = mean_rev - ref_rev
gap_pct    = 100 * jensen_gap / mean_rev if abs(mean_rev) > 1e-9 else 0.0

print(f'Day {INSPECT_DAY}')
print(f'  E[V_i]        = {mean_rev:.4f} EUR')
print(f'  V(p̄)          = {ref_rev:.4f} EUR')
print(f'  Jensen gap    = {jensen_gap:.4f} EUR  ({gap_pct:.3f}% of full)')

In [ ]:
gaps = rev_scen - ref_rev

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(
    f'Jensen Gap — Day {INSPECT_DAY}  |  '
    f'Gap = {jensen_gap:.4f} EUR  ({gap_pct:.3f}%)',
    fontsize=11, fontweight='bold'
)

# sorted per-scenario revenues
axes[0].bar(np.arange(len(rev_scen)), np.sort(rev_scen),
            color='steelblue', alpha=0.6, width=1.0,
            label='Per-scenario revenue (sorted)')
axes[0].axhline(ref_rev,  color='navy',  lw=1.5, ls='--',
                label=f'Intrinsic V(p̄) = {ref_rev:.2f} EUR')
axes[0].axhline(mean_rev, color='tomato', lw=1.5, ls='-.',
                label=f'E[V] = {mean_rev:.2f} EUR')
axes[0].set_xlabel('Scenario (sorted)'); axes[0].set_ylabel('Revenue [EUR]')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Jensen contribution histogram
axes[1].hist(gaps, bins=40, color='tomato', alpha=0.7, edgecolor='white',
             label=r'$V_i - V(\bar{p})$')
axes[1].axvline(0,            color='navy',  lw=2.0, ls='--', label='0 (intrinsic)')
axes[1].axvline(float(gaps.mean()), color='black', lw=2.0, ls='-.',
                label=f'Mean = {gaps.mean():.4f} EUR')
axes[1].set_xlabel(r'$V_i - V(\bar{p})$  [EUR]')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Jensen contributions\n'
                  'mean = Jensen gap  |  narrow & near 0 → gap negligible')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()